In [ ]:
import datasets

dataset = datasets.load_from_disk("data/msmarco_ft3")
split_dataset = dataset.train_test_split(test_size=0.25, seed=42)
pt_dataset = split_dataset['train']
pt_dataset = pt_dataset.remove_columns("scores")
ft_dataset = split_dataset['test']

ft_dataset.save_to_disk("new_data/ft/msmarco")
pt_dataset.save_to_disk("new_data/pt/msmarco")

In [6]:
import datasets
for key in ["scifact","hotpotqa","fever","nfcorpus","fiqa"]:
    dataset = datasets.load_from_disk(f"data/pt-kd-ids/{key}")
    docs_num = len(dataset["docs"][0])
    filtered_dataset = dataset.filter(lambda example: len(example["docs"])==docs_num)
    filtered_dataset = filtered_dataset.remove_columns("first_rank")
    filtered_dataset.save_to_disk(f"new_data/to_scores/{key}")
    

Saving the dataset (1/1 shards): 100%|██████████| 809/809 [00:00<00:00, 5316.95 examples/s]


Saving the dataset (1/1 shards): 100%|██████████| 5500/5500 [00:00<00:00, 6705.73 examples/s] 


In [2]:
import datasets
import numpy as np
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

def min_max_norm(x):
    min_val = x.min(axis=-1, keepdims=True)
    max_val = x.max(axis=-1, keepdims=True)
    return (x - min_val) / (max_val - min_val + 1e-8)  # 加上小量避免除零


def combine(*scores, weights=None):
    if weights is None:
        weights = [1] * len(scores)
    scores = [min_max_norm(score) * weights[i] for i, score in enumerate(scores)]
    scores = np.sum(scores, axis=0) / len(scores)
    return scores

# for dataset_name in ["nfcorpus","scifact","fever","hotpotqa","fiqa"]:
for dataset_name in ["scifact", "hotpotqa"]:

    corpus, queries, qrels = GenericDataLoader(data_folder=f"data/beir/{dataset_name}").load(
        split="train"
    )

    dataset = datasets.load_from_disk(f"new_data/to_scores/{dataset_name}")
    all_scores = []
    for identifier in ["gte-bi", "sparse-v1", "llm-3","llm-4", "t5"]:
        all_scores.append(np.load(f"new_data/scores/{dataset_name}/{identifier}_scores.npy"))

    query_to_id = {}
    for k,v in queries.items():
        text = v
        query_to_id[text] = k
        
    q_ids = []
    for q in dataset["query"]:
        q_ids.append(query_to_id[q])
        
        
    doc_to_id = {}
    for k,v in corpus.items():
        text = v["title"] + " " + v["text"]
        doc_to_id[text] = k
        
    d_ids = []
    for docs in dataset["docs"]:
        d_ids.append([doc_to_id[d] for d in docs])
        
    def eval_scores(scores):
        res = {}
        if not isinstance(scores, list):
            scores = scores.tolist()
        for i in range(len(q_ids)):
            res[q_ids[i]] = dict(zip(d_ids[i],scores[i]))

        ndcg, map_, recall, p = EvaluateRetrieval.evaluate(
            qrels, res, [1, 10]
        )

        print(ndcg)

    print(dataset_name)
    for scores in all_scores:
        eval_scores(scores)    
    eval_scores(combine(*all_scores[:-1]))


100%|██████████| 5183/5183 [00:00<00:00, 112380.34it/s]

scifact
{'NDCG@1': 0.63569, 'NDCG@10': 0.7518}
{'NDCG@1': 0.61214, 'NDCG@10': 0.72212}
{'NDCG@1': 0.64684, 'NDCG@10': 0.76144}
{'NDCG@1': 0.75713, 'NDCG@10': 0.84346}
{'NDCG@1': 0.65304, 'NDCG@10': 0.7702}
{'NDCG@1': 0.70012, 'NDCG@10': 0.8039}


100%|██████████| 5233329/5233329 [00:29<00:00, 180000.04it/s]


hotpotqa
{'NDCG@1': 0.86793, 'NDCG@10': 0.69727}
{'NDCG@1': 0.8262, 'NDCG@10': 0.70149}
{'NDCG@1': 0.91131, 'NDCG@10': 0.78794}
{'NDCG@1': 0.8648, 'NDCG@10': 0.76047}
{'NDCG@1': 0.82552, 'NDCG@10': 0.73783}
{'NDCG@1': 0.91108, 'NDCG@10': 0.78179}


In [32]:
def min_max_norm(x):
    min_val = x.min(axis=-1, keepdims=True)
    max_val = x.max(axis=-1, keepdims=True)
    return (x - min_val) / (max_val - min_val + 1e-8)  # 加上小量避免除零


def combine(*scores, weights=None):
    if weights is None:
        weights = [1] * len(scores)
    scores = [min_max_norm(score) * weights[i] for i, score in enumerate(scores)]
    scores = np.sum(scores, axis=0) / len(scores)
    return scores

def sort_docs_by_scores(examples):
    """按照scores从大到小排序docs和scores"""
    sorted_docs = []
    sorted_scores = []
    
    # 对每个样本进行排序
    for doc_list, score_list in zip(examples["docs"], examples["scores"]):
        # 将docs和scores配对并排序
        paired_data = list(zip(doc_list, score_list))
        # 根据score降序排序
        paired_data.sort(key=lambda x: x[1], reverse=True)
        
        # 解压排序后的配对数据
        docs_sorted, scores_sorted = zip(*paired_data) if paired_data else ([], [])
        
        sorted_docs.append(list(docs_sorted))
        sorted_scores.append(list(scores_sorted))
    
    # 更新examples中的数据
    examples["docs"] = sorted_docs
    examples["scores"] = sorted_scores
    
    return examples

def export(dataset_name, identifiers):
    dataset = datasets.load_from_disk(f"new_data/to_scores/{dataset_name}")
    all_scores = []
    for identifier in identifiers:
        all_scores.append(np.load(f"new_data/scores/{dataset_name}/{identifier}_scores.npy"))
    scores = combine(*all_scores) * 30
    scores = scores.tolist()
    dataset = dataset.remove_columns(["q_id", "d_ids"])
    dataset = dataset.add_column("scores", scores)
    dataset = dataset.map(
        sort_docs_by_scores, 
        batched=True, 
        desc="按分数排序文档"
    )
    dataset.save_to_disk(f"new_data/ft/{dataset_name}")